## Fill in your name below

Double-click this text cell and replace the placeholder with your full name before submitting.

**Name:** YOUR NAME HERE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session3/BMEN35_Ex3_03_Assignment_DermaMNIST_CNN.ipynb)

# Assignment 3: build and evaluate a CNN on DermaMNIST

Now you will write the code to build, train and evaluate a convolutional neural network for seven-class skin-lesion image classification.

DermaMNIST is a standardized, low-resolution version of the HAM10000 image collection. The complete dataset has 10,015 dermatoscopic images resized to 28×28 pixels, with supplied training, validation and test splits. The classes are imbalanced, and reducing the images to 28×28 removes detail. We use this small version so downloading and training are manageable in Colab.

Keep the test set for the final evaluation after you have made choices using the validation set.

## 1. Install and import the dataset tools

MedMNIST downloads the official DermaMNIST file when download=True. The 28×28 dataset is small enough for this exercise in Colab.

In [ ]:
%pip -q install medmnist

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from medmnist import DermaMNIST
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score
)

print('TensorFlow version:', tf.__version__)

## 2. Load each official split

The dataset object contains images in imgs and integer class labels in labels. Load the supplied train, validation, and test splits. Do not combine them or create a new random split. We will use training data to fit the models, validation data to compare choices, and the test data only for the final evaluation.

In [ ]:
train_data = DermaMNIST(split='train', download=True, as_rgb=True, size=28)
validation_data = DermaMNIST(split='val', download=True, as_rgb=True, size=28)
test_data = DermaMNIST(split='test', download=True, as_rgb=True, size=28)

X_train = train_data.imgs
# squeeze() removes the extra size-one dimension: one label remains per image.
y_train = train_data.labels.squeeze().astype('int64')
X_validation = validation_data.imgs
y_validation = validation_data.labels.squeeze().astype('int64')
X_test = test_data.imgs
y_test = test_data.labels.squeeze().astype('int64')

class_names = [
    'actinic keratoses', 'basal cell carcinoma', 'benign keratosis',
    'dermatofibroma', 'melanoma', 'melanocytic nevi', 'vascular lesions'
]

print('Training images:', X_train.shape, '| labels:', y_train.shape)
print('Validation images:', X_validation.shape, '| labels:', y_validation.shape)
print('Test images:', X_test.shape, '| labels:', y_test.shape)
print('Pixel range:', X_train.min(), 'to', X_train.max())

## 3. Inspect the images and class distribution

Before training, display example images and count the labels in the training split. The class codes are integers from 0 to 6; class_names maps each code to its category name.

The classes are imbalanced. Look at the counts before interpreting accuracy: a model can have a high accuracy by mostly predicting the common class.

### Stop and look: what can these images support?

1. Which class has the most training examples? Which has the fewest?
2. What details are hard to see at 28×28 pixels?
3. Why do we keep the supplied validation and test splits separate?
4. Name one reason this benchmark cannot establish performance in a real clinic.

## 4. Write a majority-class baseline

A majority-class baseline ignores image contents and always predicts the most frequent training label. Fit it using the training labels, then calculate accuracy and macro F1 on the validation split.

DummyClassifier ignores its input features, so a one-column array of zeros is enough. Do not use test labels here; we need the test split later for the final evaluation.

## 5. Prepare image values for a CNN

Each image is RGB and has shape (28, 28, 3). A batch has shape (number of images, 28, 28, 3). Convert the pixel values from integers in the range 0–255 to floating-point values in the range 0–1.

Keep the class labels as integers. Sparse categorical cross-entropy expects one integer label per image.

## 6. Define your first CNN

Write a small CNN with at least two convolution layers, at least one max-pooling layer, and a final output layer with seven softmax probabilities. Keep the network small enough for Colab. Add code comments explaining what each layer does.

## 7. Compile and train the first model

Use sparse categorical cross-entropy because each image has one integer class label. Train for 8 epochs and monitor the supplied validation split. Use the same number of epochs for the second model so the comparison is fair. Keep the test split out of this step.

## 8. Plot the first model's learning curves

Plot training and validation loss. Add a second plot for accuracy. Look for a gap between the training and validation curves; a strong training score alone does not show that the model generalizes.

## 9. Evaluate the first model on validation data

Use the validation split to calculate accuracy, macro F1, and class-wise precision, recall, and F1. This is where you compare model choices. Macro F1 gives each class equal weight, which makes it useful when the class counts differ.

## 10. Make one controlled change: weight the classes

The first model treats every training image equally. Try a second model with the same layers, optimizer, batch size, and number of epochs, but pass class weights to model.fit. This gives the rarer classes more influence on the training loss.

Calculate the weights from training labels only. Use the square root of inverse class frequency so the weighting is less extreme than using the full inverse frequency. Then compare the two models using validation macro F1. Do not use test results to choose between them.

## 11. Select a model, then evaluate the test split once

Choose between the unweighted and class-weighted CNN using validation macro F1. Then make predictions on the test split for the majority baseline and the chosen CNN. Report accuracy, macro F1, class-wise results, and a confusion matrix. Do not evaluate both CNNs on the test split.

## 12. Written interpretation

Use the class counts, learning curves, validation comparison, and final test results:

1. How did the majority-class baseline compare with the CNN on macro F1?
2. Did class weighting improve validation macro F1? Which classes gained or lost recall?
3. Which class has the lowest recall for the selected CNN? What does its confusion-matrix row show?
4. Why can test performance on this standardized low-resolution dataset differ from performance on new hospitals, cameras, or patient groups?
5. What additional data or validation would be needed before making any real-world claim?

Treat the model as a classroom exercise only; the source dataset is not intended for clinical use.

## Attribution and dataset details

DermaMNIST is part of MedMNIST and is derived from HAM10000. The full 10,015-image collection is provided in a standardized 28×28 format with a 7:1:2 train/validation/test split. DermaMNIST is licensed CC BY-NC 4.0 and is not intended for clinical use. Cite the MedMNIST dataset and the original HAM10000 paper when reusing it:

- MedMNIST: https://doi.org/10.5281/zenodo.10519652
- HAM10000: Tschandl, Rosendahl & Kittler (2018), Scientific Data 5, 180161, https://doi.org/10.1038/sdata.2018.161

## Download and submit your notebook

When you have finished the assignment:

1. Run the notebook cells from top to bottom and keep the outputs with your work.
2. In Google Colab, choose File → Download → Download .ipynb. This downloads a Jupyter notebook file with the .ipynb extension.
3. Submit the downloaded .ipynb file in Canvas.

Submit the .ipynb notebook file. Do not submit a PDF, screenshots, or a .py file.